# EfficientNet B0-B7 — Compound Model Scaling

**Paper:** Tan, M. & Le, Q. V. (2019).  
*EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.*  
International Conference on Machine Learning (ICML 2019).  
arXiv:1905.11946. [[PDF]](https://arxiv.org/abs/1905.11946)

---

## Overview

EfficientNet introduces **compound scaling**: a principled method to simultaneously scale
width, depth, and resolution of a CNN using a single compound coefficient φ.

Key observation: naively scaling only one dimension (depth OR width OR resolution) is
suboptimal.  Scaling all three in balance yields better accuracy for the same FLOP budget.

**Components:**
- **MBConv** (Mobile Inverted Bottleneck Conv): building block from MobileNetV2 + SE
- **NAS baseline (B0)**: found via Neural Architecture Search on ImageNet
- **B1-B7**: compound-scaled variants with increasing accuracy and parameter count

> Note: This is the **original EfficientNet** (B-series).  
> EfficientNetV2 (`efficientnet_v2.py`) is a later, improved version with FusedMBConv.

## Architecture (ASCII)

```
Input [B, 3, H, W]   (H=W=224 for B0)
    │
    ▼
  Stem: Conv(3→32, 3×3, s2) + BN + SiLU    → [B, 32, H/2, W/2]
    │
    ├─ Stage 1: MBConv1, k=3, s=1, out=16,  rep=1
    ├─ Stage 2: MBConv6, k=3, s=2, out=24,  rep=2  ← stride-8
    ├─ Stage 3: MBConv6, k=5, s=2, out=40,  rep=2  ← stride-16
    ├─ Stage 4: MBConv6, k=3, s=2, out=80,  rep=3
    ├─ Stage 5: MBConv6, k=5, s=1, out=112, rep=3  ← stride-16
    ├─ Stage 6: MBConv6, k=5, s=2, out=192, rep=4  ← stride-32
    └─ Stage 7: MBConv6, k=3, s=1, out=320, rep=1  ← stride-32
    │
    ▼
  Head: Conv(→1280, 1×1) + BN + SiLU → GlobalAvgPool → Dropout → [B, 1280]

MBConv6 internals:
  in → Expand 1×1(×6) → BN → SiLU
      → DW k×k         → BN → SiLU
      → SE(ratio=0.25)
      → Project 1×1    → BN
      → (+ skip if stride=1 and in==out)
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import numpy as np

BG   = '#0f1117'
FG   = '#e8eaf6'
DEPTH  = '#42a5f5'
WIDTH  = '#66bb6a'
RES    = '#ffa726'
COMP   = '#ef5350'

fig, axes = plt.subplots(1, 2, figsize=(15, 7), facecolor=BG)
fig.suptitle('EfficientNet: Compound Scaling', color=FG, fontsize=13, fontweight='bold')

# ── Left: Scaling dimensions ──────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor(BG); ax.axis('off')
ax.set_title('Three Scaling Dimensions', color=FG, fontsize=11)

variants = ['B0\n(baseline)', 'Width\nscaling', 'Depth\nscaling', 'Resolution\nscaling', 'Compound\nscaling\n(B1-B7)']
# Represent as (width, depth, res) circles at positions
cols   = [0.5, 1.7, 2.9, 4.1, 5.3]
widths = [0.5, 0.9, 0.5, 0.5, 0.85]
depths = [2,   2,   3.5, 2,   3.0]
reses  = [1.0, 1.0, 1.0, 1.5, 1.3]
colors = [FG,  WIDTH, DEPTH, RES, COMP]

for i, (name, cx, w, d, r, c) in enumerate(zip(variants, cols, widths, depths, reses, colors)):
    # Draw network layers as rectangle stack
    for layer in range(int(d)):
        rect = FancyBboxPatch((cx - w/2, layer*0.4), w*r, 0.3,
                              boxstyle='round,pad=0.02', fc=c, alpha=0.6, ec=FG, lw=0.5)
        ax.add_patch(rect)
    ax.text(cx, -0.3, name, ha='center', va='top', color=c, fontsize=7.5, fontweight='bold')

ax.set_xlim(0, 6); ax.set_ylim(-1.2, 4)
ax.text(3.0, 3.7, 'd = α^φ  w = β^φ  r = γ^φ', ha='center', color=FG, fontsize=9)
ax.text(3.0, 3.3, 'α·β²·γ² ≈ 2  (isotropic FLOP budget)', ha='center', color=FG, fontsize=8.5, alpha=0.8)

legend_handles = [
    mpatches.Patch(color=WIDTH, label='Width scaling'),
    mpatches.Patch(color=DEPTH, label='Depth scaling'),
    mpatches.Patch(color=RES,   label='Resolution scaling'),
    mpatches.Patch(color=COMP,  label='Compound (B1-B7)'),
]
ax.legend(handles=legend_handles, loc='lower right',
          facecolor=BG, edgecolor=FG, labelcolor=FG, fontsize=8)

# ── Right: MBConv block detail ────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor(BG); ax2.set_xlim(0, 6); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('MBConv6 with SE Block', color=FG, fontsize=11)

EXP  = '#42a5f5'
DW   = '#66bb6a'
SEL  = '#ab47bc'
PRJ  = '#ffa726'
SKP  = '#78909c'

def box2(y, text, color, alpha=0.7, w=4.0):
    x = (6-w)/2
    ax2.add_patch(FancyBboxPatch((x, y), w, 0.7, boxstyle='round,pad=0.05', fc=color, alpha=alpha, ec=FG, lw=1))
    ax2.text(3.0, y+0.35, text, ha='center', va='center', color=FG, fontsize=8.5)

def arr2(y):
    ax2.annotate('', xy=(3.0, y), xytext=(3.0, y+0.3), arrowprops=dict(arrowstyle='->', color=FG, lw=1.2))

box2(9.0, 'Input [B, C, H, W]', SKP)
arr2(8.3)
box2(7.6, 'Expand 1×1 (×expand_ratio) + BN + SiLU', EXP)
arr2(6.9)
box2(6.2, 'Depthwise k×k (groups=C_mid) + BN + SiLU', DW)
arr2(5.5)
box2(4.8, 'SE Block: Pool → FC(÷4) → ReLU → FC(×4) → Sigmoid × input', SEL)
arr2(4.1)
box2(3.4, 'Project 1×1 + BN (no activation)', PRJ)
arr2(2.7)

# Skip connection
ax2.plot([4.6, 5.2, 5.2, 3.0], [9.35, 9.35, 2.7, 2.7], color=SKP, lw=1, linestyle='dashed', alpha=0.7)
ax2.text(5.4, 6.0, '+\nskip', ha='center', va='center', color=SKP, fontsize=8)
ax2.text(1.5, 2.5, '(only if stride=1\nand in==out)', ha='center', color=SKP, fontsize=7.5, alpha=0.8)

box2(1.8, 'Output [B, C_out, H/s, W/s]', SKP)

plt.tight_layout()
plt.savefig('efficientnet_b_diagram.png', dpi=120, bbox_inches='tight', facecolor=BG)
plt.show()

## Key Innovations & Mathematics

### Compound Scaling Principle

Given a baseline network $\mathcal{N}$, scale width $w$, depth $d$, resolution $r$ to maximise accuracy subject to a FLOP constraint:

$$\mathcal{N}(d, w, r) = \mathcal{N}(\alpha^\phi, \beta^\phi, \gamma^\phi)$$

Subject to:
$$\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2, \quad \alpha \geq 1, \beta \geq 1, \gamma \geq 1$$

The $\beta^2$ and $\gamma^2$ terms account for the quadratic FLOP dependence on width and resolution.

B0 baseline coefficients (from NAS): $\alpha=1.2, \beta=1.1, \gamma=1.15$.

### Compound Scaling Multipliers

| Model | φ | Width | Depth | Resolution | Top-1 |
|-------|---|-------|-------|------------|-------|
| B0 | 0 | 1.0 | 1.0 | 224 | 77.1% |
| B1 | 0.5 | 1.0 | 1.1 | 240 | 79.1% |
| B2 | 1 | 1.1 | 1.2 | 260 | 80.1% |
| B3 | 2 | 1.2 | 1.4 | 300 | 81.6% |
| B4 | 3 | 1.4 | 1.8 | 380 | 82.9% |
| B7 | 6 | 2.0 | 3.1 | 600 | 84.3% |

### MBConv FLOP Cost

For MBConv with expansion ratio $t$, input channels $C_i$, output $C_o$, spatial $H \times W$:

$$\text{FLOPs} \approx H \cdot W \cdot C_i \cdot t \cdot C_i \quad \text{(expand)} + H \cdot W \cdot k^2 \cdot t \cdot C_i \quad \text{(depthwise)} + H \cdot W \cdot t \cdot C_i \cdot C_o \quad \text{(project)}$$

In [ ]:
# ── Code Walkthrough ──────────────────────────────────────────────────────────
import sys; sys.path.insert(0, '../../src')
import torch, math
from hale_core.nn.vision.cnn.efficientnet_b import MBConvB, EfficientNetB, _make_divisible

# 1. MBConvB building block
block = MBConvB(in_chs=32, out_chs=16, stride=1, expand_ratio=1, kernel_size=3)
x = torch.randn(2, 32, 56, 56)
y = block(x)
print('MBConvB(32→16, ratio=1, k=3, s=1):')
print(f'  Input:  {tuple(x.shape)}')
print(f'  Output: {tuple(y.shape)}')
print(f'  Has skip: {block.has_skip}')

# 2. Compound scaling
dm, wm = 1.4, 1.2  # B3 coefficients
print('\nB3 scaling (B0 stage 4: out=80, blocks=3):')
scaled_out = _make_divisible(80 * wm, 8)
scaled_blk = max(1, math.ceil(3 * dm))
print(f'  Width-scaled channels: {scaled_out}')
print(f'  Depth-scaled blocks:   {scaled_blk}')

In [ ]:
# ── Usage Example ─────────────────────────────────────────────────────────────
from hale_core.nn.vision import build_vision_backbone

print('EfficientNet-B series backbone outputs:')
for name in ['efficientnet_b0', 'efficientnet_b3', 'efficientnet_b7']:
    b = build_vision_backbone(name)
    b.eval()
    x = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        out = b(x)
    print(f'\n{name}:')
    print(f'  embed_dim: {b.embed_dim}')
    print(f'  pooled:    {tuple(out.pooled_output.shape)}')
    print(f'  feature_maps: {[tuple(f.shape) for f in out.feature_maps]}')
    print(f'  params:    {b.num_parameters():,}')

## References

| # | Citation |
|---|----------|
| [1] | Tan, M. & Le, Q. V. (2019). **EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks.** ICML. arXiv:1905.11946. |
| [2] | Tan, M. & Le, Q. V. (2021). **EfficientNetV2: Smaller Models and Faster Training.** ICML. arXiv:2104.00298. |
| [3] | Sandler, M. et al. (2018). **MobileNetV2: Inverted Residuals and Linear Bottlenecks.** CVPR. arXiv:1801.04381. |
| [4] | Zoph, B. et al. (2018). **Learning Transferable Architectures for Scalable Image Recognition (NASNet).** CVPR. |
| [5] | Hu, J. et al. (2018). **Squeeze-and-Excitation Networks.** CVPR. arXiv:1709.01507. |
| [6] | Huang, G. et al. (2016). **Deep Networks with Stochastic Depth.** ECCV. |

---
*Notebook: EfficientNet B0-B7 — HaleBlocks backbone family*  
*Architecture: `hale_core.nn.vision.cnn.efficientnet_b`*